# Tabela Silver — `ecommerce_enderecos`

Este notebook aplica regras de qualidade usando **PySpark**, separa registros válidos e rejeitados, e envia ambos para tabelas separadas no SQL Server.

A tabela Silver contém todos os registros, com colunas de auditoria:

- `invalidado`: `Sim` ou `Não`
- `is_valido`: `S` ou `N`
- `motivo_rejeicao`
- `data_hora_rejeicao`

Assim, registros inválidos não são descartados e também não é necessário criar uma tabela separada de rejeitados.

In [0]:
%run ../utils/utils

## Imports e parâmetros

In [0]:



import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from functools import reduce
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_enderecos"
TABELA_DQ = "dq_monitoring_logs"

print(f"Iniciando processamento Silver - Endereços - Run ID: {RUN_ID}")

## Anti-Join e Tabelas de Referência

In [0]:
# 1. Carrega a tabela Bronze de enderecos
try:
    df_bronze_enderecos = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    
    # Verifica na nova subpasta de quarentena
    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_endereco") \
            .union(df_quarentena.select("id_endereco"))
    else:
        df_processados = df_silver_atual.select("id_endereco")
        
    df_micro_lote = df_bronze_enderecos.join(df_processados, "id_endereco", "left_anti")
else:
    df_micro_lote = df_bronze_enderecos

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência (Carrega df_clientes_ref de forma correta)
# =================================================================================
if delta_existe("silver", "ecommerce_clientes", STORAGE_OPTIONS):
    df_clientes_ref = ler_delta("silver", "ecommerce_clientes", STORAGE_OPTIONS) \
        .select(F.col("id_cliente").alias("id_cliente_ref")).dropDuplicates()
else:
    schema = StructType([StructField("id_cliente_ref", LongType(), True)])
    df_clientes_ref = spark.createDataFrame([], schema)

print("Tabela de referência df_clientes_ref carregada com sucesso.")

In [0]:
print("Colunas disponíveis no df_micro_lote:", df_micro_lote.columns)

## Aplicação das 10 Regras de Qualidade

In [0]:
if qtd_novos > 0:
    # Parâmetros de Validação
    regex_cep = r"^\d{8}$"
    ufs_validas = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
    apelidos_validos = ['Casa', 'Trabalho', 'Outro']
    
    # Janelas Analíticas para Regras de Negócio
    w_id_endereco = Window.partitionBy("id_endereco")
    w_id_cliente = Window.partitionBy("id_cliente")

    # Prepara os dados (Limpeza, Normalização e Agregação)
    df_base = df_micro_lote \
        .join(df_clientes_ref, df_micro_lote.id_cliente == df_clientes_ref.id_cliente_ref, "left") \
        .withColumn("cep_limpo", F.regexp_replace(F.col("cep").cast("string"), r"[^0-9]", "")) \
        .withColumn("lat_num", F.col("latitude").cast("double")) \
        .withColumn("lon_num", F.col("longitude").cast("double")) \
        .withColumn("estado_norm", F.upper(F.trim(F.col("estado")))) \
        .withColumn("apelido_norm", F.initcap(F.trim(F.col("apelido")))) \
        .withColumn("is_principal_seguro", F.col("is_principal").cast("string").isin("true", "True", "1", "S", "Sim", "Y", "yes")) \
        .withColumn("qtd_id_endereco", F.count("*").over(w_id_endereco)) \
        .withColumn("qtd_enderecos_cliente", F.count("*").over(w_id_cliente)) \
        .withColumn("qtd_principal_cliente", F.sum(F.when(F.col("is_principal_seguro"), 1).otherwise(0)).over(w_id_cliente))

    # Aplicação das 10 Regras de Endereços
    df_silver_enderecos = df_base \
        .withColumn("r1_id_endereco_falhou", F.col("id_endereco").isNull() | (F.trim(F.col("id_endereco").cast("string")) == "") | (F.col("qtd_id_endereco") > 1)) \
        .withColumn("r2_id_cliente_fk_falhou", F.col("id_cliente").isNull() | F.col("id_cliente_ref").isNull()) \
        .withColumn("r3_cep_falhou", F.col("cep_limpo").isNull() | (~F.col("cep_limpo").rlike(regex_cep))) \
        .withColumn("r4_estado_uf_falhou", F.col("estado_norm").isNull() | (~F.col("estado_norm").isin(ufs_validas))) \
        .withColumn("r5_coordenadas_br_falhou", F.col("lat_num").isNull() | F.col("lon_num").isNull() | (F.col("lat_num") < -33.75) | (F.col("lat_num") > 5.27) | (F.col("lon_num") < -73.99) | (F.col("lon_num") > -32.39)) \
        .withColumn("r6_unico_principal_falhou", F.col("qtd_principal_cliente") != 1) \
        .withColumn("r7_apelido_padrao_falhou", F.col("apelido_norm").isNotNull() & (F.trim(F.col("apelido_norm")) != "") & (~F.col("apelido_norm").isin(apelidos_validos))) \
        .withColumn("r8_limite_enderecos_falhou", F.col("qtd_enderecos_cliente") > 3) \
        .withColumn("r9_logradouro_falhou", F.col("logradouro").isNull() | (F.trim(F.col("logradouro")) == "")) \
        .withColumn("r10_numero_falhou", F.col("numero").isNull() | (F.trim(F.col("numero").cast("string")) == ""))

    print("Muralha de qualidade de endereços estruturada com sucesso.")
else:
    print("Etapa ignorada: não há micro-lote novo.")


## Catalogo de Regras e Logs

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_endereco_falhou", "regra": "R1_ID_ENDERECO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_CLIENTE_FK_ORFAO", "severidade": "Critica"},
        {"coluna": "r3_cep_falhou", "regra": "R3_CEP_FORMATO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_estado_uf_falhou", "regra": "R4_UF_INVALIDA", "severidade": "Critica"},
        {"coluna": "r5_coordenadas_br_falhou", "regra": "R5_COORDENADAS_FORA_DO_BRASIL", "severidade": "Aviso"},
        {"coluna": "r6_unico_principal_falhou", "regra": "R6_CLIENTE_SEM_ENDERECO_PRINCIPAL_UNICO", "severidade": "Critica"},
        {"coluna": "r7_apelido_padrao_falhou", "regra": "R7_APELIDO_FORA_DO_PADRAO", "severidade": "Aviso"},
        {"coluna": "r8_limite_enderecos_falhou", "regra": "R8_CLIENTE_COM_MAIS_DE_3_ENDERECOS", "severidade": "Aviso"},
        {"coluna": "r9_logradouro_falhou", "regra": "R9_LOGRADOURO_VAZIO", "severidade": "Critica"},
        {"coluna": "r10_numero_falhou", "regra": "R10_NUMERO_VAZIO", "severidade": "Critica"}
    ]

    total_registros = df_silver_enderecos.count()
    logs_list = []
    
    for r in catalogo_regras:
        qtd_falhas = df_silver_enderecos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, TABELA_ALVO, r["regra"], "FAIL", r["severidade"],
                int(qtd_falhas), int(total_registros), datetime.now(timezone.utc), f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    # --- AJUSTE: Separação dinâmica e unificação das condições ---
    flags_criticas = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Critica"]
    flags_avisos = [r["coluna"] for r in catalogo_regras if r["severidade"] == "Aviso"]

    condicao_invalida_critica = reduce(lambda a, b: a | b, [F.col(c) for c in flags_criticas])
    condicao_aviso = reduce(lambda a, b: a | b, [F.col(c) for c in flags_avisos]) if flags_avisos else F.lit(False)

    # UNIFICAÇÃO: Se for Crítica OU Aviso, a linha não é válida para a Silver
    condicao_total_falha = condicao_invalida_critica | condicao_aviso

    df_silver_enderecos = df_silver_enderecos \
        .withColumn("silver_linha_valida", ~condicao_total_falha) \
        .withColumn("silver_tem_aviso", condicao_aviso) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))

    print("Logs de qualidade estruturados. Avisos agora também invalidam a linha para a Silver.")
else:
    # Garantia de existência da variável caso o lote seja vazio
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Etapa ignorada: não há micro-lote novo.")

## Gravar Silver válida e logs

In [0]:
if qtd_novos > 0:
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id", "silver_tem_aviso"]
    
    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    # Certifique-se de que df_silver_enderecos é a variável correta carregada acima
    df_silver_validos = df_silver_enderecos \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = df_silver_enderecos \
        .filter(F.col("silver_linha_valida") == False) \
        .select(*colunas_finais)
        
    # Lógica única de gravação de quarentena
    if df_silver_invalidos.count() > 0:
        # A. Verifica histórico para deduplicar
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            # Dedup pelo ID da sua tabela (ex: id_cliente ou id_rastreamento)
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_cliente"), 
                on="id_cliente", 
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        # B. Grava apenas o que sobrou (o que não existe no histórico)
        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar,
                camada="silver/quarentena",
                tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False 
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    # Garante que logs vazios não quebrem o processamento
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos, 
            camada="", 
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, 
            mode="append", 
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  Gravar Silver e `dq_monitoring_logs`

In [0]:
#  confirmação da criação e salvamento
display(
    spark.table("squad1.dq_monitoring_logs")
    .orderBy(F.col("timestamp_execucao").desc())
)

##  Validação final e visualização das tabelas

In [0]:
print("===== VALIDAÇÃO FINAL =====")

# 1. Validação da tabela Silver principal
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe no Data Lake.")

# 2. Validação da tabela de Logs de Qualidade (na raiz do Data Lake)
if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    
    # Filtra para mostrar apenas os logs referentes à tabela de endereços
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")